<a href="https://colab.research.google.com/github/amahashabde/GenAI-SimpleStockAnalysis/blob/main/AM_LangGraph_StockAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Problem Statement
Investors and financial enthusiasts frequently seek quick insights about stock performance, including current price, trading range, valuation metrics, and market activity. However, obtaining this information typically requires navigating multiple financial platforms, interpreting raw numerical data, and performing manual analysis.

This process can be time-consuming, fragmented, and cognitively demanding, particularly for beginner investors who may lack the expertise to interpret financial indicators such as P/E ratios, trading volume, or 52-week price ranges.

The problem addressed in this project is to design an AI-driven automated stock analysis system capable of:

Understanding natural language user queries

Identifying the correct stock ticker symbol

Fetching live stock market data

Generating human-readable analytical insights

In [3]:
!pip install -q langgraph langchain langchain-openai yfinance

import os
import yfinance as yf
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from google.colab import userdata

# -------------------------------------------------
# API KEY
# -------------------------------------------------
api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# -------------------------------------------------
# LLM
# -------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# -------------------------------------------------
# STATE
# -------------------------------------------------
class StockState(TypedDict, total=False):
    user_query: str
    ticker: str
    stock_data: dict
    analysis: str
    next_step: str

# -------------------------------------------------
# AGENT NODE: decides what to do next
# -------------------------------------------------
def agent_decide(state: StockState) -> StockState:
    prompt = f"""
You are a simple stock assistant agent.

Current state:
User Query: {state.get('user_query', '')}
Ticker: {state.get('ticker', '')}
Stock Data Present: {'yes' if state.get('stock_data') else 'no'}
Analysis Present: {'yes' if state.get('analysis') else 'no'}

Decide the NEXT step.

Rules:
- If ticker is missing, return only: extract
- Else if stock_data is missing, return only: fetch
- Else if analysis is missing, return only: analyze
- Else return only: finish

Return exactly one word from:
extract
fetch
analyze
finish
"""
    decision = llm.invoke(prompt).content.strip().lower()

    allowed = {"extract", "fetch", "analyze", "finish"}
    if decision not in allowed:
        decision = "finish"

    return {**state, "next_step": decision}

# -------------------------------------------------
# NODE 1: Extract ticker
# -------------------------------------------------
def extract_ticker(state: StockState) -> StockState:
    prompt = f"""
Extract the stock ticker symbol only from this query.

Examples:
Apple -> AAPL
Tesla -> TSLA
Microsoft -> MSFT

Query: {state['user_query']}

Return only the ticker symbol.
"""
    ticker = llm.invoke(prompt).content.strip().upper()
    return {**state, "ticker": ticker}

# -------------------------------------------------
# NODE 2: Fetch stock price
# -------------------------------------------------
def fetch_stock_price(state: StockState) -> StockState:
    try:
        info = yf.Ticker(state["ticker"]).info

        stock_data = {
            "company": info.get("longName", "N/A"),
            "current_price": info.get("currentPrice", "N/A"),
            "day_high": info.get("dayHigh", "N/A"),
            "day_low": info.get("dayLow", "N/A"),
            "52w_high": info.get("fiftyTwoWeekHigh", "N/A"),
            "52w_low": info.get("fiftyTwoWeekLow", "N/A"),
            "pe_ratio": info.get("trailingPE", "N/A"),
            "volume": info.get("volume", "N/A"),
        }

        return {**state, "stock_data": stock_data}

    except Exception as e:
        return {
            **state,
            "stock_data": {
                "company": "N/A",
                "current_price": "N/A",
                "day_high": "N/A",
                "day_low": "N/A",
                "52w_high": "N/A",
                "52w_low": "N/A",
                "pe_ratio": "N/A",
                "volume": "N/A",
                "error": str(e)
            }
        }

# -------------------------------------------------
# NODE 3: Analyze stock
# -------------------------------------------------
def analyze_stock(state: StockState) -> StockState:
    d = state["stock_data"]

    prompt = f"""
You are a stock analyst.
Give a very simple 5-line analysis for a beginner.

Company: {d['company']}
Current Price: {d['current_price']}
Day High: {d['day_high']}
Day Low: {d['day_low']}
52 Week High: {d['52w_high']}
52 Week Low: {d['52w_low']}
P/E Ratio: {d['pe_ratio']}
Volume: {d['volume']}

Keep it simple, clear, and practical.
"""
    analysis = llm.invoke(prompt).content
    return {**state, "analysis": analysis}

# -------------------------------------------------
# ROUTER
# -------------------------------------------------
def route_next_step(state: StockState) -> Literal["extract", "fetch", "analyze", "finish"]:
    return state["next_step"]

# -------------------------------------------------
# BUILD GRAPH
# -------------------------------------------------
builder = StateGraph(StockState)

builder.add_node("agent", agent_decide)
builder.add_node("extract", extract_ticker)
builder.add_node("fetch", fetch_stock_price)
builder.add_node("analyze", analyze_stock)

builder.set_entry_point("agent")

builder.add_conditional_edges(
    "agent",
    route_next_step,
    {
        "extract": "extract",
        "fetch": "fetch",
        "analyze": "analyze",
        "finish": END,
    }
)

# after each action, go back to the agent
builder.add_edge("extract", "agent")
builder.add_edge("fetch", "agent")
builder.add_edge("analyze", "agent")

graph = builder.compile()

# -------------------------------------------------
# RUN IT
# -------------------------------------------------
queries = [
    "How is Apple stock doing today?",
    "What is Tesla's current price?",
    "Give me an update on Microsoft",
]

for query in queries:
    result = graph.invoke({
        "user_query": query,
        "ticker": "",
        "stock_data": {},
        "analysis": "",
        "next_step": ""
    })

    d = result["stock_data"]

    print("=" * 60)
    print("User Query:", query)
    print("Ticker:", result.get("ticker", "N/A"))
    print(f"{d.get('company', 'N/A')} | ${d.get('current_price', 'N/A')}")
    print(f"High: ${d.get('day_high', 'N/A')}  Low: ${d.get('day_low', 'N/A')}")
    print(f"52W: ${d.get('52w_high', 'N/A')} / ${d.get('52w_low', 'N/A')}")
    print(f"P/E: {d.get('pe_ratio', 'N/A')} | Volume: {d.get('volume', 'N/A')}")
    print()
    print(result.get("analysis", "No analysis generated"))
    print("=" * 60)

User Query: How is Apple stock doing today?
Ticker: AAPL
Apple Inc. | $273.17
High: $273.74  Low: $266.87
52W: $288.62 / $193.25
P/E: 34.534767 | Volume: 42729584


User Query: What is Tesla's current price?
Ticker: TSLA
Tesla, Inc. | $387.51
High: $392.99  Low: $385.3
52W: $498.83 / $244.43
P/E: 358.80554 | Volume: 53758836


User Query: Give me an update on Microsoft
Ticker: MSFT
Microsoft Corporation | $432.92
High: $433.64  Low: $423.67
52W: $555.45 / $356.28
P/E: 27.074423 | Volume: 27224146


